# 2025년 5월 27일 화요일 (23일차)



<br><br><br>

# 📜 목차
- DB연동 처음부터 다시 해보기
  - DB 연동 클래스 만들기
    - 주로 활용하는 **DB '접근' 및 '조회'**관련 코드 class로 만들기
    - 만들어놓은 **DB '접근' 및 '조회' class 코드** 사용해보기
- DB 연동 클래스 체계를 Score문제에 적용해보기
- 커넥션 풀
  - 커넥션 풀 score 문제로 만들기
  - 커넥션 풀 weekly_pay 문제로 만들기

<br><br><br><br>

# 🟩 DB연동 처음부터 다시 해보기


## 🟢 MySQL DB Table 만들기

In [ ]:
CREATE DATABASE project1;

USE project1;

-- ---------------------------
-- 1. 회원 테이블 만들기 
CREATE TABLE tb_member( 
			-- mysql은 auto_increment 속성있는 필드가 무조건 primary key가 되어야 한다.
			member_id bigint AUTO_INCREMENT PRIMARY KEY,
			user_id varchar(40),
			password varchar(300),  -- md5 암호화 알고리즘을 써서 암호화 해서 저장.
																												-- 저장
			user_name varchar(40),
			email varchar(40),
			phone varchar(40),
			regdate datetime
);

DROP table  tb_member;

INSERT INTO tb_member(user_id, password, user_name, email, phone, regdate)
values('test1', '1234', '홍길동', 'hong@daum.net', '010-0000-0001', now());

SELECT * FROM tb_member;

## 🟢 python 코드 DB 조작하기

### 🟡 소켓(Socket) 에 대해서 
- 통신을 담당하는 라이브러리
- 컴퓨터 네트워크에서 통신의 끝점을 나타내는 개념입니다.
- 즉, 두 컴퓨터가 데이터를 주고받기 위해 서로 연결되는 출입구라고 이해하면 됩니다.

- 소켓에 부여된 번호가 port이다.

- 2byte 정수 1 ~ 65535까지 가능하다.
- 1 ~ 1000 은 함부로 못씀

- 80 - http(웹서버)
  - www.daum.net:80  (80이 모두 기본 default 입니다.)

- 21 - telnet

- 22 - ssh

- 23 - FTP


### 🟡 주로 활용하는 **DB '접근' 및 '조회'**관련 코드 class로 만들기

In [ ]:
# python_workspace4 폴더를 만들고,
# DBModule.py 파일을 만듭니다.

import pymysql
import pymysql.cursors


class Database:
    def __init__(self):
        self.conn = self.mysql_conn()
        self.cursor = self.conn.cursor(pymysql.cursors.DictCursor)

    # init 해줄것을 이렇게 만들었습니다.
    def mysql_conn(self):
        conn = pymysql.connect(
            host="localhost",
            user="root",
            password="",
            db="project1",
            port=3306,
        )
        print("접속 성공")
        return conn

    # execute = 실행하다.
    # insert, update, delete 할 때 사용할 수 있게 이렇게 만들었다.
    def execute(self, query, args=()):
        # args - tuple 기본값
        print(args)
        self.cursor.execute(query, args)
        self.conn.commit()

    # 데이터 딱 1개만 가져오기
    # scalar 쿼리 포함,  select count(*) from tb_member
    def executeOne(self, query, args=()):
        self.cursor.execute(query, args)
        row = self.cursor.fetchone()
        return row  # 결과를 반환해야한다. 첫번째 레코드값 하나만 가져간다.

    # 데이터 여러개 가져오기
    def executeAll(self, query, args=()):
        self.cursor.execute(query, args)
        rows = self.cursor.fetchall()
        return rows

    # 닫기
    def close(self):
        if self.conn.open:
            self.conn.close

    

### 🟡 만들어놓은 **DB '접근' 및 '조회' class 코드** 사용해보기

In [ ]:
# useDBModule.py 라는 별도의 파일을 만들었습니다.
# from DBModule import Database  # import하여 사용하겠습니다.

# cursor(커서)는 DB와 연결된 상태에서 SQL을 실행하고, 결과를 가져오는 객체

db = Database()  # 객체 만들면 이미 디비 접근
sql = "select * from tb_member"
rows = db.excuteAll(sql)
for row in rows:
    print(row)
db.close()  # 원래는 열었다 닫았다 하는 것이 맞습니다.

접속 성공
{'member_id': 1, 'user_id': 'test1', 'password': '1234', 'user_name': '홍길동', 'email': 'hong@daum.net', 'phone': '010-0000-0001', 'regdate': datetime.datetime(2025, 5, 27, 10, 5, 40)}


### 🟡 만들어놓은 **DB '조작' class 코드** 만들기

In [ ]:
class SignUp:
    def __init__(self):
        # project1 DB를 사용
        self.db = Database()  # 객체 만들면 디비 접근
        
    # 1. 전체 보기
    def all_view_data(self):
        sql = """
            SELECT
                member_id, user_id, password, 
                user_name, email, phone, regdate,
                date_format(regdate, '%Y-%m-%d %H:%i') regdate
            FROM tb_member;
        """
        # print(sql)
        self.db.cursor.execute(sql)
        rows = self.db.cursor.fetchall()
        print("데이터 개수", len(rows))
        for row in rows:
            print(
                row["member_id"],
                row["user_id"],
                row["password"],
                row["user_name"],
                row["email"],
                row["phone"],
                row["regdate"],
            )
        print()

    # ID vaildation
    def validate_id(self, id):
        sql = """
            SELECT
                user_id, password, user_name, email, phone, regdate,
                date_format(regdate, '%Y-%m-%d %H:%i') regdate
            FROM tb_member;
        """
        self.db.cursor.execute(sql)
        rows = self.db.cursor.fetchall()
        for row in rows:
            if row["user_id"] == id:
                return True
        return False
    
    # 함께 만들어본 ID vaildation 
    def idcheck(self, user_id=""):
        if user_id == "":
            return False   # 사용 불가
        sql = "select count(*) cnt from tb_member where user_id = %s"
            # 🔥🔥🔥 sql 명령어로 데이터를 다 처리해서 가져올 생각을 해야합니다.
        row = self.db.executeOne(sql, (user_id))
        self.db.close()
        if row["cnt"] == 0:
            return True  #중복 안되었다면 쓸 수 있다.
        return False

    # 2. 회원가입 (Insert Data)
    def member_register(self):
        user_id = input("아이디 : ")
        if self.validate_id(user_id) == True:
            return print("🚫 이미 존재하는 ID입니다. 다시 실행해주세요.")
        else:
            print("🟢 사용할 수 있는 ID입니다.")
            password = input("패스워드 : ")
            user_name = input("이름 : ")
            email = input("이메일 : ")
            phone = input("전화 : ")
            sql = """
                INSERT INTO tb_member(user_id, password, user_name, email, phone, regdate)
                values(%s, %s, %s, %s, %s, now())
            """
            self.db.execute(sql, (user_id, password, user_name, email, phone))  # 여기 안의 괄호는 tuple을 의미합니다.
            print("\n----------- 📝 가입 현황 파악 -----------")
            self.all_view_data()
            print()

    # 3. 데이터 수정
    def update_data(self):
        # 수정 전 전체 리스트 파악
        print("\n----------- ✅ 수정할 데이터를 선택하세요 -----------")
        self.all_view_data()
        # input으로 입력받기
        member_id = input("수정할 member_id 입력 : ")
        user_id = input("ID : ")
        password = input("Password : ")
        user_name = input("Name : ")
        email = input("e-mail : ")
        phone = input("Phone : ")
        sql = """
            UPDATE tb_member
            SET 
                user_id = %s,
                password = %s,
                user_name = %s,
                email = %s,
                phone = %s
            WHERE member_id = %s
        """
        self.db.cursor.execute(
            sql, (user_id, password, user_name, email, phone, member_id)
        )
        print("수정 완료")
        print()
        print("\n\n----------- 📝 가입 현황 파악 -----------")
        self.all_view_data()
        print()

    # 4. 데이터 삭제
    def delete_data(self):
        # 삭제 전 전체 현재 데이터 파악
        print("----------- ✅ 삭제할 데이터를 선택하세요 -----------")
        self.all_view_data()
        member_id = input("삭제할 member_id 입력 : ")
        sql = """
            DELETE FROM tb_member WHERE member_id = %s
        """
        self.db.cursor.execute(sql, member_id)
        print("삭제 완료")
        print("\n\n----------- 📝 가입 현황 파악 -----------")
        self.all_view_data()
        print()

    # 5. 메뉴 선택형
    def start(self):
        while True:
            print(f"1.전체가입현황보기  |  2.회원가입  |  3.  |  4.  |  0.종료")
            select = input("🔢 번호 선택: ")

            if select == "1": 
                self.all_view_data()    # 1. 전체가입현황보기
            elif select == "2":
                self.member_register()  # 2. 회원가입
            elif select == "3":
                self.update_data()      # 3. 수정
            elif select == "4":
                self.delete_data()      # 4. 삭제
            elif select == "0":
                break

if __name__ == "__main__":
    m = SignUp()
    m.start()




접속 성공
1.전체가입현황보기  |  2.회원가입  |  3.  |  4.  |  0.종료


데이터 개수 3
1 test1 1234 홍길동 hong@daum.net 010-0000-0001 2025-05-27 10:05:40
2 3 1234 임꺽정 Eeee@gmail.com 010-000-0000 2025-05-27 11:22:50
3 2 1234 피카츄 peka@gmail.com 010-0000-0000 2025-05-27 11:31:36

1.전체가입현황보기  |  2.회원가입  |  3.  |  4.  |  0.종료
데이터 개수 3
1 test1 1234 홍길동 hong@daum.net 010-0000-0001 2025-05-27 10:05:40
2 3 1234 임꺽정 Eeee@gmail.com 010-000-0000 2025-05-27 11:22:50
3 2 1234 피카츄 peka@gmail.com 010-0000-0000 2025-05-27 11:31:36

삭제 완료

----------- 📝 가입 현황 파악 -----------
데이터 개수 2
1 test1 1234 홍길동 hong@daum.net 010-0000-0001 2025-05-27 10:05:40
3 2 1234 피카츄 peka@gmail.com 010-0000-0000 2025-05-27 11:31:36


1.전체가입현황보기  |  2.회원가입  |  3.  |  4.  |  0.종료
1.전체가입현황보기  |  2.회원가입  |  3.  |  4.  |  0.종료
1.전체가입현황보기  |  2.회원가입  |  3.  |  4.  |  0.종료
